## this contains the implementation of various excercises questions for video 1

### common imports & global values

In [1]:
from dataclasses import dataclass,field
import numpy as np
import torch
import torch.nn.functional as F
from abc import ABC, abstractmethod
from itertools import islice

In [2]:

@dataclass
class ModelConfigs:
    """This class contains certain hyperparameter values that can be experimented upoin
    
        Class Members
            LEARNING_RATE: int - the step size taken during the optimization in gradient based approach
            REGURLARIZATION_VALUE: float - the loss factor on applied explicitly on the weights (L1/L2) regularization
            SMOOTHING_COUNT: int - make the counts array (counting approach) away from zero so the logs make more sense (away from inf probabilities)
            BOUNDARY_CHAR: str - start and end of the sequence
            SEED_VAL: int - pseudo random generator seed for reproducible samples.
    """
    LEARNING_RATE:int = field(default=50)
    REGULARIZATION_VALUE:float = field(default=0.01)
    SMOOTHING_COUNT: int = field(default=1)
    BOUNDARY_CHAR: str = field(default=".")
    START_BOUNDARY_CHAR: str = field(default=".")
    END_BOUNDARY_CHAR: str = field(default=".")
    SEED_VAL: int = field(default=2147483647)

In [3]:
@dataclass
class Vocab:
    """Vocab is a packaging class which contains important information for language modelling tasks

        Members:
            boundary_char: str - this signifies the start and end of a sequence
            vocab_letters: list[str] - this signifies the list of unique characters that occur in the data set
            stoi: dict[str, int] - this is the mapping between the the unique characters and int values
            itos: dict[int, str] -  this is the reverse of stoi
            n_unique: int - this is the number of unique characters + the boundary character
    """
    start_boundary_char: str
    end_boundary_char: str
    vocab_letters: list[str] = field(default_factory=list)
    stoi: dict[str, int] = field(default_factory=dict)
    itos: dict[int, str] = field(default_factory=dict)
    n_unique: int = field(default=0)


In [4]:
@dataclass
class NGram:
    """Packing class that contains the information related to the NGram classes

        Class Members
            data: list[tuple[str, ...]] - pairs of ngrams identified from within the sequence of strings
            n: length of the example subset (n = 2) means bigram, (n = 3) means trigram and so on.
    """
    data: list[tuple[str, ...]] = field(default_factory=list)
    n: int = field(default=2)

In [5]:
def get_names(data_path: str = '../names.txt')->list[str]:
    """Read the names file and load the contentents into memory as a list"""
    with open(data_path, 'r') as file:
        names =  file.read().splitlines()
        names = list(set(names))
    return names

In [6]:
def create_vocab(start_boundary_char: str, end_boundary_char: str, data_set: list[str])->Vocab:
  """Creates a vocab object for the provided boundary character and list of strings


    Args:
      start_boundary_char: str - the start of a sequence
      end_boundary_char : str - the end of a sequence
      data_set: list[str] - list of character sequences that are to be modelled
    
    Returns
      vocab: Vocab - this is the vocab object that contains important character level model information.
  """


  # create a list of unique occuring characters from the set
  unique_chars = sorted(list(set(''.join(data_set))))
  
  # add boundary char at index 0
  unique_chars.insert(0, start_boundary_char)

  # create the string to int mapping for the unqiue characters
  stoi = {
    s: i
    for i, s in enumerate(unique_chars)
  }

  # reverse the above mapping to have int -> string
  itos = {
    i : s
    for s, i in stoi.items()
  }


  # package all together as a object to be used in other parts of the program
  return Vocab(
    start_boundary_char = start_boundary_char,
    end_boundary_char = end_boundary_char,
    vocab_letters= unique_chars[1:],
    stoi = stoi,
    itos=itos,
    n_unique=len(unique_chars)
  )
  



In [7]:
def generate_n_grams(data_set: list[str], start_boundary_char: str, end_boundary_char: str,n: int = 2)->NGram:
    """Function to generate the n-gram set of the input names dataset by default it generates bigram examples

    Args:
        data_set: list[str] = this is a list of strings (names)
        start_boundary_char : str = this is appended at the start of the string
        end_boundary_char: str = this is the appended at the end of the string

        n: int = if 2 we generate bigrams, 3 we generate tri-grams etc.

    Returns:
        n_grams = list[tupe[str,...]] -> this is a list of tuple each tuple will atleast have two elements (the input character and the resultant character) in case of bigrams
    """

    # resultant list
    n_grams = []

    for name in data_set:
        
        # append the boundary char
        char_list = [start_boundary_char] + list(name) + [end_boundary_char]

        for i in range(len(char_list) - n + 1):
            n_grams.append(tuple(char_list[i : i + n]))
    return NGram(
        n = n,
        data=n_grams
    )


In [8]:
class NGramInterface(ABC):
    """Common Interface for a count based NGram implementation

        Object Members
            data_set:list[str] - list of sequences to model
            vocab:Vocab - object containing information about the ngram models vocabulary
            n_gram:NGram - information about the n_gram
            counts: torch.tensor - counts of the ngram occuring in all of the sequences.
            probs: torch.tensor - normalized counts (each row sums up to 1.) probability if the next occuring character given a n-1 context
            generator: torch.Generator - a generator object to be used across all the ngram impplementations for reproducible results
    
    """
    def __init__(self, data_set: list[str], n: int, config: ModelConfigs)->None:
        self.config = config
        self.data_set = data_set
        self.vocab = create_vocab(data_set=data_set, start_boundary_char=config.START_BOUNDARY_CHAR, end_boundary_char=config.END_BOUNDARY_CHAR)
        self.n_gram = generate_n_grams(data_set= data_set, start_boundary_char=config.START_BOUNDARY_CHAR,end_boundary_char=config.END_BOUNDARY_CHAR, n = n)
        self.counts = self.create_counts_array()
        self.probs = self.counts.float()
        self.probs /= self.probs.sum(dim=-1, keepdim=True)
        self.generator = torch.Generator().manual_seed(config.SEED_VAL)
    
    def create_counts_array(self)->torch.Tensor:
        """Common method to generate the counts array for an n-gram model
        """
        # use laplace smoothing from the start
        counts = torch.full(size=(self.vocab.n_unique, )*self.n_gram.n, fill_value=self.config.SMOOTHING_COUNT, dtype=torch.int32)
        n = self.n_gram.n

        # traverse through all the ngrams over all the sequences
        for data in self.n_gram.data:
            # get the indices
            indices = [self.vocab.stoi[ch] for ch in islice(data, n)]
            # increment by 1
            counts[*indices] += 1
        # return the counts
        return counts
    
    
    def calculate_nll_counting_method(self)->None:
        """Common method for all ngrams to calculate the nll loss. the lower the better
        """
        # keep a running sum of the individual log probs
        log_likelihood = 0
        n = self.n_gram.n
        # iterate over all the bigrams over the data set
        for data in self.n_gram.data:
            # get the int mapping of the characters
            indices = [self.vocab.stoi[ch] for ch in islice(data, n)]
            # get the probability assigned for the bigram pair
            p = self.probs[*indices]
            # get the log of the value
            log_prob = p.log()
            # accumualate the log prob sum
            log_likelihood += log_prob.item()
        # invert the sign so we can use a loss value
        neg_log_likelihood = -log_likelihood

        # normalize to have a easily interpretable value
        neg_log_likelihood /= len(self.n_gram.data)
        # display to the user
        print(f"{neg_log_likelihood=:.4f}")
    
    
    
    def sample_names_using_counting_method(self, sample_size: int = 5, max_length: int = 5)->None:
        """Common logic to generate sequences using the ngram counting approach
            Args:
                sample_size: int - number of sequences to be generated.
                max_length: int - number of tokens in a sequence.
            Returns: None, we just print the sequences one by one.
        """

        for _ in range(sample_size):
            out = []
            context = [self.vocab.stoi[self.vocab.start_boundary_char]] * (self.n_gram.n - 1)

            while True:
                p = self.probs[tuple(context)]
                next_ix = torch.multinomial(p, num_samples=1, replacement=True, generator=self.generator).item()
                context = context[1:] + [next_ix]
                out.append(self.vocab.itos[next_ix])
                if next_ix == self.vocab.stoi[self.vocab.end_boundary_char]:
                    break
                if len(out) > max_length:
                    out.append(self.vocab.end_boundary_char)
                    break

            print(''.join(out))
    
    

In [9]:
class Bigram(NGramInterface):
    """Concerte implementation of the ngram interface for modelling a bigram model using the counting and / or gradient based approach.
        Methods
            __init__ : returns None calls the super class constructor to create the vocabulary and ngram data and set seed
            create_counts_array: provides the implementation for counting bigrams and set the count member variable inherited by the parent interface which inturn creates the probs tensor
            sample_names_using_counting_method : for a set sample size defaults to 5 generate random samples based on the statistics learned via counting approach
            calculate_nll_counting_method: calculate the negative log likelihood of the bigram model implemented using the counting approach.
    """
    def __init__(self, data_set: list[str], config:ModelConfigs)->None:
        super().__init__(data_set=data_set,n = 2, config=config)
        self.config = config




In [10]:
class Trigram(NGramInterface):
    """Concerte implementation of the ngram interface for modelling a trigram model using the counting and / or gradient based approach.
        Methods
            __init__ : returns None calls the super class constructor to create the vocabulary and ngram data and set seed
            create_counts_array: provides the implementation for counting bigrams and set the count member variable inherited by the parent interface which inturn creates the probs tensor
            sample_names_using_counting_method : for a set sample size defaults to 5 generate random samples based on the statistics learned via counting approach
            calculate_nll_counting_method: calculate the negative log likelihood of the bigram model implemented using the counting approach.
    """
    def __init__(self, data_set: list[str], config:ModelConfigs):
        super().__init__(data_set=data_set,  n = 3, config=config)
        self.config = config

        

In [11]:
# get the names
names = get_names(data_path='../names.txt')
names_mixed = get_names(data_path='../names_mixed.txt')


In [12]:
config = ModelConfigs()

In [13]:
# genrerate the bigram and trigram objects
bigrams = Bigram(data_set=names, config=config)
trigrams = Trigram(data_set=names, config=config)

In [14]:
bigrams_mixed = Bigram(data_set=names_mixed, config=config)
trigrams_mixed = Trigram(data_set=names_mixed, config=config)

In [15]:
bigrams.sample_names_using_counting_method(), print("*"*25), bigrams_mixed.sample_names_using_counting_method()

dexze.
momasu.
railez.
kaynn.
konimi.
*************************
cexza.
comaku.
rarori.
kay.
kerini.


(None, None, None)

In [16]:
trigrams.sample_names_using_counting_method(), print("*"*25), trigrams_mixed.sample_names_using_counting_method()

ce.
za.
zogh.
uriana.
za.
*************************
cexzm.
zoglku.
railez.
kayhwm.
vinimj.


(None, None, None)

In [17]:
bigrams.calculate_nll_counting_method(), bigrams_mixed.calculate_nll_counting_method()

neg_log_likelihood=2.4543
neg_log_likelihood=2.4161


(None, None)

In [18]:
trigrams.calculate_nll_counting_method(), trigrams_mixed.calculate_nll_counting_method()

neg_log_likelihood=2.0999
neg_log_likelihood=2.1798


(None, None)